# Fine-tuning médical — TinyLlama 1.1B

**Projet :** TechCorp IA — Mission Expérimentale R&D  
**Modèle :** `TinyLlama/TinyLlama-1.1B-Chat-v1.0`  
**Dataset :** `ruslanmv/ai-medical-chatbot` (HuggingFace)  
**Méthode :** LoRA via HuggingFace PEFT  

> ⚠️ Ce modèle est expérimental — pas pour production médicale.

## Étape 1 — Installation des dépendances

In [ ]:
!pip install -q transformers==4.45.2 peft==0.12.0 bitsandbytes==0.43.3 \
    datasets accelerate trl matplotlib

## Étape 2 — Vérification du GPU

In [ ]:
import torch

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ Aucun GPU détecté — active le GPU dans Exécution > Modifier le type d'exécution")

## Étape 3 — Chargement du modèle TinyLlama

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"✅ Modèle chargé : {MODEL_NAME}")
print(f"Paramètres : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Étape 4 — Configuration LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Étape 5 — Chargement et préparation du dataset médical

In [ ]:
from datasets import load_dataset

NUM_SAMPLES = 2000

raw_dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
print(f"Dataset complet : {len(raw_dataset)} exemples")
print(f"Colonnes : {raw_dataset.column_names}")

dataset = raw_dataset.shuffle(seed=42).select(range(NUM_SAMPLES))
print(f"Sous-ensemble utilisé : {len(dataset)} exemples")
print("\nExemple :")
print(f"Patient : {dataset[0]['Patient'][:150]}")
print(f"Doctor  : {dataset[0]['Doctor'][:150]}")

In [ ]:
SYSTEM_PROMPT = "You are a helpful medical assistant. Provide clear, accurate and cautious medical information."

def format_conversation(example):
    text = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{example['Patient']}</s>\n"
        f"<|assistant|>\n{example['Doctor']}</s>"
    )
    return {"text": text}

dataset = dataset.map(format_conversation, remove_columns=dataset.column_names)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"Train : {len(train_dataset)} | Eval : {len(eval_dataset)}")
print("\nExemple formaté :")
print(train_dataset[0]["text"][:400])

## Étape 6 — Entraînement

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./tinyllama-medical-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
)

print("🚀 Démarrage de l'entraînement...")
train_result = trainer.train()
print("✅ Entraînement terminé !")

## Étape 7 — Métriques et courbe de loss

In [ ]:
import matplotlib.pyplot as plt

logs = [log for log in trainer.state.log_history if "loss" in log]
steps = [l["step"] for l in logs]
losses = [l["loss"] for l in logs]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, label="Training Loss", color="steelblue")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("TinyLlama — Fine-tuning médical (LoRA)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

print("\n=== Résumé de l'entraînement ===")
print(f"Loss initiale  : {losses[0]:.4f}")
print(f"Loss finale    : {losses[-1]:.4f}")
print(f"Amélioration   : {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")
print(f"Steps total    : {train_result.global_step}")
print(f"Temps total    : {train_result.metrics['train_runtime']:.0f}s")

## Étape 8 — Test du modèle fine-tuné

In [ ]:
TEST_QUESTIONS = [
    "What are the symptoms of diabetes?",
    "I have a headache and fever since 2 days, what should I do?",
    "What is the difference between a cold and the flu?",
]

model.eval()
for question in TEST_QUESTIONS:
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{question}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {response.strip()}")
    print("-" * 60)

## Étape 9 — Sauvegarde du modèle

In [ ]:
trainer.save_model("./tinyllama-medical-lora")
tokenizer.save_pretrained("./tinyllama-medical-lora")
print("✅ Modèle sauvegardé dans ./tinyllama-medical-lora")
print("\n📋 Pour partager : File > Share dans Colab, puis copiez le lien.")